<a href="https://colab.research.google.com/github/Nbassey01/Machine_Failure_Prediction/blob/main/MLS_2_Session_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

## Business Context

In modern manufacturing, predictive maintenance plays a critical role in ensuring equipment availability and minimizing unplanned downtime. Unexpected machine failures can cause significant delays and financial losses. A manufacturing company is working to enhance the efficiency and reliability of its predictive maintenance system by leveraging operational and sensor data collected from machine tools. This data includes variables such as air temperature, process temperature, rotational speed, torque, tool wear, and failure indicators.

While the company has developed a machine learning model to predict failures, the current process for data handling, model training, and deployment is manual and time-consuming—requiring engineers to rerun notebooks each time new data becomes available. This introduces inefficiencies and delays in responding to potential failures.

To address this, there is a clear need for an MLOps pipeline that can seamlessly handle data registration, preprocessing, model training with tuning, and deployment. Such a pipeline would enable real-time integration of new data, reduce manual effort, and support timely, data-driven maintenance decisions—ultimately improving production reliability and reducing operational costs.

## Objective

You have been hired as an MLOps engineer, and your task is to design and develop an end-to-end MLOps pipeline that automates key machine learning tasks such as data registration, preprocessing, model training with hyperparameter tuning, and deployment to production. The goal is to eliminate the need for manually executing each component in a notebook whenever new data arrives, thereby reducing manual effort and improving operational efficiency

## Pre-requisites

* Create a Github repo
    - Go to ***Github Profile***
    - Click on ***Your repositories*** then select ***New***
      - Repository Name: ***Machine_Failure_Prediction***
      - Check the box ***README.md*** file
      - Click on ***Create repository***

* Adding hugging face space secrets to Github Actions to execute the workflow
  1. Go to Hugging Face ***Profile***
  2. Navigate to ***Access Token***
  3. Create a ***New token***
      - Token type ***Write***
      - Token Name ***MLOps***
      - Click on ***Create Token***
      - Copy the generated Token
  4. Now, go to Github repo
      - Click on ***Settings***
      - Navigate to ***Secrets and Variables***
      - Click on ***Actions***
      - Add a ***Repository secerts***
        - Name ***HF_TOKEN***
        - Secret: ***Paste the token created from the hugging face access tokens***
        - Click on ***Add secret***

* Create a Hugging Face space
    - Go to **Hugging Face**
    - Open your **Profile**
    - Click on **New Space**
      - Under the space creation, enter the below details
        - Space name: **Machine-Failure-Prediction**
    (If you were trying with different names, be cautious when using a underscore `_` in space names, such as `frontend_space`, as it can cause exceptions when accessing the API URL. Always use an hyphen `-` instead, like `frontend-space`.)
        - Select the space SDK: **Docker**
        - Choose a Docker template: **Streamlit**
        - Click on **Create Space**

In [14]:
!pip install -q PyGithub==2.9.1

In [15]:
import os
from github import Github, GithubException

In [16]:
# Edit these three values, then run every cell top to bottom. Nbassey01/Machine_Failure_Prediction
GITHUB_USERNAME  ="Nbassey01" # Your GitHub username
REPO_NAME        ="Machine_Failure_Prediction" # Repositiory name
COLAB_SECRET_NAME = "Github_clss_token_3" # Name of the secret in colabs

REPO  = f"{GITHUB_USERNAME}/{REPO_NAME}"
BRANCH = "main"

In [17]:
# Make the token available to both PyGithub and the gh CLI
import os
from google.colab import userdata
os.environ["GH_TOKEN"] = userdata.get(COLAB_SECRET_NAME)
print("Token loaded.")

Token loaded.


In [18]:
for d in ["project/data", "project/model_building", "project/deployment",
                   "project/.github/workflows"]:
    os.makedirs(d, exist_ok=True)

print("Folders ready.")

Folders ready.


# Model Building

## Data Registration

Once the **data** folder created after executing the above cell, please upload the **machine-failure-prediction.csv** in to the folder

In [19]:
%%writefile project/model_building/data_register.py
import pandas as pd

RAW_PATH = "data/machine-failure-prediction.csv"

# Load the raw dataset
df = pd.read_csv(RAW_PATH)

#Validate that the expected columns are present before registering it
expected_columns = [
    "UDI", "Type", "Air temperature", "Process temperature",
    "Rotational speed", "Torque", "Tool wear", "failure",
]
missing = [c for c in expected_columns if c not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing expected columns: {missing}")

print("Dataset registered succesfully.")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("Columns:", list(df.columns))
print("Failure distribution:")
print(df["failure"].value_counts())

Overwriting project/model_building/data_register.py


## Data Preparation

In [20]:
%%writefile project/model_building/prep.py
# for data manipulation
import pandas as pd
from sklearn.model_selection import train_test_split

"
df = pd.read_csv("/content/project/data/machine-failure-prediction.csv")
df.drop(columns=['UDI'], inplace=True)# Drop the unique identifier

# NOTE:'Type' is intentionally left as raw strings (H/L/M).
# The training pipeline one-hot-encodes it, and the streamlit app also sends
#raw H/L/M values. Encoding it here (e.g. LabelEncoder) would make training
# and serving use differnet representations, silently breaking predictions.

# Split into X (features) and y (target)
X = df.drop(columns=["Failure"])
y = df["Failure"]

# Perform train-test split
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Xtrain.to_csv("Xtrain.csv",index=False)
Xtest.to_csv("Xtest.csv",index=False)
ytrain.to_csv("ytrain.csv",index=False)
ytest.to_csv("ytest.csv",index=False)

print("Data prepared: train /testsplits written.")
print("Type values kept as:", sorted(X["Type"].unique()))

Overwriting project/model_building/prep.py


## Model Training

In [21]:
%%writefile project/model_building/train.py
# for data manipulation
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
# for model serialization
import joblib

Xtrain = pd.read_csv("Xtrain_path")
Xtest = pd.read_csv("Xtest_path")
ytrain = pd.read_csv("ytrain_path").squeeze()
ytest = pd.read_csv("ytest_path").squeeze()

# One-hot encode 'Type' and scale numeric features
numeric_features = [
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear'
]
categorical_features = ['Type']


# Class weight to handle imbalance
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]

# Preprocessing pipeline
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=class_weight, random_state=42)

# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 75, 100],
    'xgbclassifier__max_depth': [2, 3],
    'xgbclassifier__learning_rate': [0.05, 0.1],
}

# Create pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

# Grid search with cross-validation
grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_search.fit(Xtrain, ytrain)

# Best model
best_model = grid_search.best_estimator_
print("Best Params:\n", grid.best_params_)
print(classification_report(ytest, best_model.predict(Xtest))

# Save best model
joblib.dump(best_model, "deployment/best_machine_failure_model_v1.joblib")
print("Model save to deployment/best_machine_failure_model_v1.joblib")


Overwriting project/model_building/train.py


# Deployment

## Dockerfile

## Streamlit App

In [22]:
%%writefile project/deployment/app.py
import os
import streamlit as st
import pandas as pd
import joblib

# Download and load the model
model_path = os.path.join(os.path.dirname(_file_),"best_machine_failure_model_v1.joblib")
model = joblib.load(model_path)

# Streamlit UI for Machine Failure Prediction
st.title("Machine Failure Prediction App - Nsikak")
st.write("""
This application predicts the likelihood of a machine failing based on its operational parameters.
Please enter the sensor and configuration data below to get a prediction.
""")

# User input
Type = st.selectbox("Machine Type", ["H", "L", "M"])
air_temp = st.number_input("Air Temperature (K)", min_value=250.0, max_value=400.0, value=298.0, step=0.1)
process_temp = st.number_input("Process Temperature (K)", min_value=250.0, max_value=500.0, value=324.0, step=0.1)
rot_speed = st.number_input("Rotational Speed (RPM)", min_value=0, max_value=3000, value=1400)
torque = st.number_input("Torque (Nm)", min_value=0.0, max_value=100.0, value=40.0, step=0.1)
tool_wear = st.number_input("Tool Wear (min)", min_value=0, max_value=300, value=10)

# Assemble input into DataFrame
input_data = pd.DataFrame([{
    'Air temperature': air_temp,
    'Process temperature': process_temp,
    'Rotational speed': rot_speed,
    'Torque': torque,
    'Tool wear': tool_wear,
    'Type': Type
}])


if st.button("Predict Failure"):
    prediction = model.predict(input_data)[0]
    result = "Machine Failure" if prediction == 1 else "No Failure"
    st.subheader("Prediction Result:")
    st.success(f"The model predicts: **{result}**")

Overwriting project/deployment/app.py


## App Dependency Handling

In [23]:
%%writefile project/deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4

Overwriting project/deployment/requirements.txt


# Create MLOps pipeline with Github Action Workflow

## Action Workflow YAML File

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Action Workflow

In [24]:
%%writefile project/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4

Overwriting project/requirements.txt


Work Flow or YAML file

In [25]:
%%writefile project/.github/workflows/pipeline.yml
name: Machine Failure MLOps Pipeline

on:
  workflow_dispatch:      #Lets you click "Run workflow" in the Action tab

permissions:
  content: write

jobs:
  # ---------- JOB 1:Register Dataset ----------
  register-dataset:
    name: Register
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: pip install -r project/requirements.txt
      - name: Register dataset
        run: python model_building/data_register.py
      - name: Upload registered dataset
        uses: actions/upload-artifact@v4
        with:
          name: registered-data
          path: data/machine-failure-prediction.csv

  # ---------- JOB 2: Data Preparation ----------
  data-preparation:
    name: Data Preparation
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: pip install -r project/requirements.txt
      - name: Prepare data
        run: python model_building/prep.py
      - name: Upload train/test splits
        uses: actions/upload-artifact@v4
        with:
          name: data-splits
          path: |
            Xtrain.csv
            Xtest.csv
            ytrain.csv
            ytest.csv

   # ---------- JOB 3: Model Training ----------
  model-training:
    name: Model Training
    needs: data-preparation           # waits for job 2 to finish
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: pip install -r project/requirements.txt
      - name: Download train/test splits
        uses: actions/download-artifact@v4
        with:
          name: data-splits          # brings Xtraib/Xtest/ytrain/ytest from job 2
      - name: Train model
        run: python model_building/train.py
      - name: Commit trained model

        run:
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add deployment/best_machine_failure_model_v1.joblib
          git commit -m "Add trained model [skip ci]" || echo "No changes to commit"
          git push


Overwriting project/.github/workflows/pipeline.yml


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [26]:
# --- Safety check: the dataset must ne uploaded locally before pushing ---
CSV_LOCAL = "project/data/machine-failure-prediction.csv"
assert os.path.exists(CSV_LOCAL), (
    f"{CSV_LOCAL} not found. Upload machine-failure-prediction.csv into the "
    "project/data/ folder (Colab file browser, left panel) before running this cell."
)

gh   = Github(os.environ["GH_TOKEN"])
user = gh.get_user()

try:
    repo = gh.get_repo(REPO)
    print("Repo already exists:", repo.full_name)
except GithubException:
    repo = user.create_repo(
        REPO_NAME,
        private=False,    # set True if you want it porivate
        auto_init=True,   # creates README so the mail branch exists
    )
    print("repo created:", repo.full_name)
    # --- Clean up any stray nested 'week_2_mls'/ folder from earlier runs ---
    # files must live at the repo ROOT (e.g. model_building/prep.py),
    # NOT nested (week_2_mls/model_buidling/prep.py). this removes old nested copies.
    def delete_path(path):
        try:
          contents = repo.get_contents(path, ref=BRANCH)
        except GithubException:
            return # nothing there
        for item in contents:
            if item.type =="dir":
                delete_path(item.path)
            else:
                repo.delete_file(item.path, f"cleanup {item.path}", item.sha, branch=BRANCH)
                print("removed stray", item.path)

    delete_path("project")

    # --- Rush local week_2_mls/ contents to the repo ROOT ---
    def push_folder(local_dir):
        for root, _, files in os.walk(local_dir):
            for fname in files:
                local_path = os.path.join(root, fname)
                repo_path  = os.path.relpath(local_path, local_dir) # strip 'week_2_mls/'
                content    = open (local_path, "rb").read()
                try:
                   sha = repo.get_contents(repo_path, ref=BRANCH).sha
                   repo.update_file(repo_path, f"update {repo_path}", content, sha, branch=BRANCH)
                   print("updated", repo_path)
                except GithubException:
                   repo.create_file(repo_path, f"add {repo_path}", content, branch=BRANCH)
                   print("added ", repo_path)

    push_folder("project")

    # --- verify the key files landed at the repo ROOT ---
    print("\nverifying repo layout...")
    required = [
        "requirements.txt",
        "model_building/registry.py",
        "model_building/prep.py",
        "model_building/train.py",
        "data/machine-failure-prediction.csv",
        "github/workflows/pipeline.yml",
        "deployment/app.py",
        "deployment/requirements.txt",
    ]
    all_ok = True
    for f in required:
        try:
           repo.get_contents(f, ref=BRANCH)
           print("  ok    ", f)
        except GithubException:
           print("  MISSING", f); all_ok = False

    print("\nALL files pushed." if all_ok else "\nSome files are missing - check the list above.")

/tmp/ipykernel_3847/3489478059.py:8: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  gh   = Github(os.environ["GH_TOKEN"])


Repo already exists: Nbassey01/Machine_Failure_Prediction


<font size=6 color="navyblue">Power Ahead!</font>
___